# Khipus.ai
## Módulo 5 · Unidad 1
### Demo: embeddings y similitud semántica con Microsoft Foundry

<span>© Copyright Notice 2025, Khipus.ai - All Rights Reserved.</span>

Adaptación a Microsoft Foundry: 8 de octubre de 2026. Se utiliza el despliegue existente **text-embedding-3-large**.

## Objetivos
Un embedding representa un texto como un vector numérico. Aprenderás a generar uno, comprobar sus dimensiones y comparar textos mediante similitud coseno.

Esta práctica utiliza **3072 dimensiones por defecto**, correspondientes a `text-embedding-3-large`. La versión anterior enseñaba Ada con 1536 dimensiones. No mezcles vectores de modelos distintos, aunque tengan el mismo tamaño.

Ejecuta las celdas en orden. Se realizan **dos solicitudes de embeddings**: una para un texto y otra por lote para los documentos y la consulta. No se genera una respuesta de chat.

## Antes de empezar: preparar el entorno virtual
Estos tres notebooks comparten el único archivo [`requirements.txt`](requirements.txt) de esta carpeta. En PowerShell, desde esta carpeta, crea el entorno si aún no existe y actívalo:

```powershell
python -m venv .venv
.\.venv\Scripts\Activate.ps1
```

Si todavía no instalaste las dependencias (o cambió `requirements.txt`), instálalas una sola vez:

```powershell
python -m pip install -r requirements.txt
```

En VS Code selecciona `.venv` como intérprete y kernel. Si ya preparaste el entorno y los requirements, continúa sin repetir la instalación. La comparación usa la biblioteca estándar; no requiere NumPy ni una base de datos vectorial.

In [ ]:
# EN CASO DE NO HABER CREADO EL .VENV
# python -m venv .venv

UsageError: Line magic function `%python` not found (But cell magic `%%python` exists, did you mean that instead?).


In [ ]:
# ACTIVAR EL ENTORNO VIRTUAL
# Windows
# .venv\Scripts\activate
# macOS / Linux
# source .venv/bin/activate

In [ ]:
# INSTALAR LAS DEPENDENCIAS
# pip install -r requirements.txt

## Configurar la conexión
Define los valores en la celda de constantes y ejecútala antes de crear el cliente:
- `AZURE_OPENAI_API_KEY`: clave del recurso proporcionada por el instructor.
- `AZURE_OPENAI_API_BASE`: endpoint Azure OpenAI terminado en `/openai/v1`.
- `AZURE_OPENAI_DEPLOYMENT`: nombre del despliegue que usará esta práctica.

La conexión y las solicitudes toman sus valores de estas constantes. No se leen variables de entorno ni se solicita una contraseña interactiva.

La ruta **Azure OpenAI v1** ya incluye la versión; no se necesita `AZURE_OPENAI_API_VERSION`. El Project Endpoint (`/api/projects/...`) no se utiliza para estas llamadas. La configuración no envía solicitudes; la primera llamada a la API comprobará tus credenciales.

In [ ]:
from urllib.parse import urlparse

from openai import OpenAI

In [ ]:
AZURE_OPENAI_API_KEY = "YOUR_AZURE_API_KEY"
AZURE_OPENAI_API_BASE = "https://khipus-ai-dev-chmrq-resource.openai.azure.com/openai/v1"
AZURE_OPENAI_DEPLOYMENT = "text-embedding-3-large"

In [ ]:
base_url = AZURE_OPENAI_API_BASE.strip().rstrip("/") + "/"

url = urlparse(base_url)
if url.scheme != "https" or not url.netloc or url.path != "/openai/v1/":
    raise ValueError("AZURE_OPENAI_API_BASE debe ser el endpoint HTTPS que termina en /openai/v1/.")

if not AZURE_OPENAI_API_KEY.strip():
    raise ValueError("Completa AZURE_OPENAI_API_KEY en la celda de constantes antes de continuar.")

client = OpenAI(
    api_key=AZURE_OPENAI_API_KEY.strip(),
    base_url=base_url,
    timeout=60.0,
    max_retries=2,
)
print("Cliente configurado. La primera solicitud comprobará la conexión.")

In [ ]:
embedding_deployment = AZURE_OPENAI_DEPLOYMENT.strip()
if not embedding_deployment:
    raise ValueError("AZURE_OPENAI_DEPLOYMENT no puede estar vacía.")
print("Despliegue de embeddings:", embedding_deployment)

## 1. Generar un embedding
Se envía un texto al despliegue de embeddings. La respuesta contiene un vector, no una explicación en lenguaje natural. Se muestran solo sus primeros valores para no llenar el notebook con miles de números.

In [ ]:
example_text = "Los modelos de lenguaje pueden ayudar a resumir documentos."

embedding_response = client.embeddings.create(
    model=embedding_deployment,
    input=example_text,
    encoding_format="float",
)

vector = embedding_response.data[0].embedding
print("Texto:", example_text)
print("Dimensiones observadas:", len(vector))
print("Primeros 8 valores:", [round(value, 6) for value in vector[:8]])
if embedding_response.usage is not None:
    print("Tokens de entrada:", embedding_response.usage.prompt_tokens)

## 2. Preparar una búsqueda pequeña
La consulta y los documentos se convierten con **el mismo modelo** en una sola solicitud. Antes de ejecutarla, predice cuál documento debería ser el más relacionado con la consulta.

In [ ]:
documents = [
    "Los embeddings permiten encontrar documentos con un significado similar a una consulta.",
    "Un modelo de lenguaje genera texto a partir de instrucciones y contexto.",
    "La fotosíntesis permite a las plantas transformar la energía de la luz.",
    "Las redes neuronales se entrenan ajustando sus parámetros a partir de ejemplos.",
]
query = "¿Cómo puedo buscar textos por su significado y no solo por palabras exactas?"

texts = documents + [query]
batch_response = client.embeddings.create(
    model=embedding_deployment,
    input=texts,
    encoding_format="float",
)

# index vincula cada vector con la posición del texto enviado.
ordered_items = sorted(batch_response.data, key=lambda item: item.index)
if [item.index for item in ordered_items] != list(range(len(texts))):
    raise RuntimeError("La respuesta no contiene un vector por cada texto enviado.")

vectors = [item.embedding for item in ordered_items]
document_vectors = vectors[:-1]
query_vector = vectors[-1]
dimensions = {len(item) for item in vectors}
if len(dimensions) != 1 or not query_vector:
    raise RuntimeError("Los vectores deben ser no vacíos y tener el mismo tamaño.")

print("Documentos:", len(documents))
print("Vectores recibidos:", len(vectors))
print("Dimensiones:", len(query_vector))
if batch_response.usage is not None:
    print("Tokens del lote:", batch_response.usage.prompt_tokens)

## 3. Calcular la similitud coseno
La similitud coseno compara la orientación de dos vectores:

$$\operatorname{sim}(a,b)=\frac{a\cdot b}{\lVert a\rVert\lVert b\rVert}$$

Un valor mayor indica mayor cercanía según este modelo. **No es una probabilidad ni un porcentaje de verdad**, y no existe un umbral universal para todas las aplicaciones.

In [ ]:
from math import sqrt

def cosine_similarity(a, b):
    if len(a) != len(b) or not a:
        raise ValueError("Los vectores deben tener igual tamaño y no estar vacíos.")
    norm_a = sqrt(sum(value * value for value in a))
    norm_b = sqrt(sum(value * value for value in b))
    if norm_a == 0 or norm_b == 0:
        raise ValueError("La similitud coseno no está definida para un vector nulo.")
    return sum(x * y for x, y in zip(a, b)) / (norm_a * norm_b)

ranking = sorted(
    [
        (cosine_similarity(query_vector, doc_vector), document)
        for document, doc_vector in zip(documents, document_vectors)
    ],
    key=lambda item: item[0],
    reverse=True,
)

print("Consulta:", query, "\n")
for position, (score, document) in enumerate(ranking, start=1):
    print(f"{position}. Similitud: {score:.4f} | {document}")

## Actividades del estudiante
1. Explica por qué el documento con más palabras compartidas no siempre es el más cercano semánticamente.
2. Sustituye la consulta por “¿De dónde obtienen energía las plantas?” y ejecuta nuevamente la solicitud por lote y el ranking.
3. Añade dos documentos de tu elección y repite la búsqueda. Justifica el nuevo orden.
4. Explica la diferencia entre recuperar documentos con embeddings y generar una respuesta con un LLM.

**Observaciones:** registra la consulta, el documento mejor situado y tu interpretación. Cada repetición del lote genera consumo adicional; recalcular solo el ranking no llama a la API.

**Extensión opcional:** los modelos `text-embedding-3` permiten solicitar un tamaño menor con `dimensions=1024`. Si experimentas con ello, añade el parámetro a ambas llamadas, regenera los vectores y compara los rankings. No mezcles tamaños.

## Referencias y conexión
- [Guía de la unidad y solución de errores](README.md).
- [Embeddings con Azure OpenAI](https://learn.microsoft.com/en-us/azure/foundry/openai/how-to/embeddings).
- [Dimensiones y comparación de embeddings](https://developers.openai.com/api/docs/guides/embeddings).

Para errores 401, 404 o 429, revisa respectivamente la clave, el nombre del despliegue/endpoint y la cuota. Al terminar, apaga el kernel.